# Intelligent Asset Lifecycle Management — Model 3
## Lifecycle Decision Engine

This notebook combines the outputs of:

- **Model 1:** Failure probability
- **Model 2:** Remaining Useful Life (RUL)

with operational/business information:

- Asset criticality
- Repair cost
- Hours since maintenance
- Previous failures

The result is an explainable lifecycle recommendation:

**MONITOR / SCHEDULE MAINTENANCE / URGENT MAINTENANCE / REPAIR / REPLACE**

> This is a rule-based decision engine, not another ML model. That makes the recommendation transparent and easy to explain during the hackathon.


In [ ]:
# ============================================================
# 1. IMPORTS AND MODEL PATHS
# ============================================================

import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

MODEL_1_PATH = "random_forest_asset_failure_model.joblib"

# Change this filename if you saved Model 2 with a different name.
MODEL_2_PATH = "random_forest_rul_model.joblib"

print("Model 1 exists:", os.path.exists(MODEL_1_PATH))
print("Model 2 exists:", os.path.exists(MODEL_2_PATH))


## 2. Save Model 2 First

In the **Model 2 notebook**, after selecting the best model, run:

```python
import joblib

joblib.dump(best_model, "random_forest_rul_model.joblib")
```

Then place both `.joblib` files in the same folder as this notebook.

Model 1 should already be saved as:

`random_forest_asset_failure_model.joblib`


In [ ]:
# ============================================================
# 3. LOAD BOTH MODELS
# ============================================================

if not os.path.exists(MODEL_1_PATH):
    raise FileNotFoundError(
        f"Model 1 not found: {MODEL_1_PATH}"
    )

if not os.path.exists(MODEL_2_PATH):
    raise FileNotFoundError(
        f"Model 2 not found: {MODEL_2_PATH}\n"
        "Save best_model from the Model 2 notebook first."
    )

failure_model = joblib.load(MODEL_1_PATH)
rul_model = joblib.load(MODEL_2_PATH)

print("Both models loaded successfully.")


## 4. Lifecycle Decision Logic

The engine uses two ML outputs:

### Failure probability
How likely the asset is to fail within 24 hours.

### RUL
Estimated remaining operating life in hours.

It then adjusts the recommendation using asset criticality, maintenance history, and repair cost.

The thresholds are intentionally explicit so they can be changed later according to the organization's risk tolerance.


In [ ]:
# ============================================================
# 5. DECISION ENGINE
# ============================================================

def lifecycle_decision(
    failure_probability,
    rul_hours,
    criticality="Medium",
    repair_cost=0,
    hours_since_maintenance=0,
    previous_failures=0
):
    """
    Return a transparent lifecycle recommendation and explanation.
    """

    # Normalize values
    failure_probability = float(np.clip(failure_probability, 0, 1))
    rul_hours = max(float(rul_hours), 0)
    repair_cost = max(float(repair_cost), 0)
    hours_since_maintenance = max(float(hours_since_maintenance), 0)
    previous_failures = max(int(previous_failures), 0)

    criticality = str(criticality).strip().lower()

    # --------------------------------------------------------
    # Risk score: 0-100
    # --------------------------------------------------------
    failure_score = failure_probability * 50

    if rul_hours <= 24:
        rul_score = 30
    elif rul_hours <= 48:
        rul_score = 24
    elif rul_hours <= 72:
        rul_score = 18
    elif rul_hours <= 168:
        rul_score = 10
    else:
        rul_score = 0

    criticality_score = {
        "low": 0,
        "medium": 5,
        "high": 10,
        "critical": 15
    }.get(criticality, 5)

    history_score = min(
        5,
        (hours_since_maintenance / 500) * 3
        + previous_failures * 0.75
    )

    risk_score = min(
        100,
        failure_score + rul_score + criticality_score + history_score
    )

    # --------------------------------------------------------
    # Primary action
    # --------------------------------------------------------
    if failure_probability >= 0.75 and rul_hours <= 24:
        action = "URGENT MAINTENANCE"
        reason = (
            "Very high failure probability and less than 24 hours "
            "of predicted useful life."
        )

    elif failure_probability >= 0.60 and rul_hours <= 48:
        action = "SCHEDULE MAINTENANCE"
        reason = (
            "High failure risk combined with short predicted RUL."
        )

    elif criticality in ["high", "critical"] and (
        failure_probability >= 0.50 or rul_hours <= 48
    ):
        action = "PRIORITY INSPECTION"
        reason = (
            "The asset is operationally important and shows "
            "elevated failure/RUL risk."
        )

    elif failure_probability >= 0.50:
        action = "MONITOR CLOSELY"
        reason = (
            "Failure probability is elevated, but immediate "
            "intervention is not yet indicated."
        )

    elif rul_hours <= 24:
        action = "INSPECT ASSET"
        reason = (
            "Predicted useful life is less than 24 hours."
        )

    else:
        action = "NORMAL OPERATION"
        reason = (
            "Current failure probability and predicted RUL "
            "do not indicate immediate action."
        )

    # --------------------------------------------------------
    # Repair vs replacement consideration
    # --------------------------------------------------------
    replacement_flag = False

    # This is deliberately a configurable heuristic rather than
    # a hard economic rule.
    if (
        repair_cost >= 100000
        and criticality == "low"
        and (
            failure_probability >= 0.60
            or rul_hours <= 48
        )
    ):
        replacement_flag = True
        action = "EVALUATE REPLACEMENT"
        reason = (
            "High repair exposure combined with low asset "
            "criticality and elevated lifecycle risk."
        )

    return {
        "risk_score": round(risk_score, 2),
        "action": action,
        "replacement_flag": replacement_flag,
        "reason": reason
    }


## 6. Test the Decision Engine

Before connecting it to live model predictions, test several hypothetical scenarios.


In [ ]:
# ============================================================
# 7. HYPOTHETICAL SCENARIOS
# ============================================================

scenarios = pd.DataFrame([
    {
        "Asset": "Machine A",
        "failure_probability": 0.82,
        "rul_hours": 14,
        "criticality": "High",
        "repair_cost": 35000,
        "hours_since_maintenance": 420,
        "previous_failures": 2
    },
    {
        "Asset": "Machine B",
        "failure_probability": 0.12,
        "rul_hours": 76,
        "criticality": "Medium",
        "repair_cost": 15000,
        "hours_since_maintenance": 120,
        "previous_failures": 0
    },
    {
        "Asset": "Machine C",
        "failure_probability": 0.65,
        "rul_hours": 8,
        "criticality": "Low",
        "repair_cost": 120000,
        "hours_since_maintenance": 600,
        "previous_failures": 4
    }
])

results = []

for _, row in scenarios.iterrows():
    result = lifecycle_decision(
        failure_probability=row["failure_probability"],
        rul_hours=row["rul_hours"],
        criticality=row["criticality"],
        repair_cost=row["repair_cost"],
        hours_since_maintenance=row["hours_since_maintenance"],
        previous_failures=row["previous_failures"]
    )

    results.append({
        "Asset": row["Asset"],
        "Failure Probability": row["failure_probability"],
        "Predicted RUL (hours)": row["rul_hours"],
        "Criticality": row["criticality"],
        "Risk Score": result["risk_score"],
        "Recommended Action": result["action"],
        "Reason": result["reason"]
    })

scenario_results = pd.DataFrame(results)

display(scenario_results)


## 7. Connect the Engine to the Two ML Models

The input below represents one asset's current sensor/operational state.

**Important:** use the same feature columns and preprocessing assumptions used when training Model 1 and Model 2.

The saved pipelines handle their own preprocessing.


In [ ]:
# ============================================================
# 8. PREDICT FROM A REAL ASSET ROW
# ============================================================

# Load the original dataset used for training.
DATA_PATH = "your_dataset.csv"  # <-- change this

if os.path.exists(DATA_PATH):
    data = pd.read_csv(DATA_PATH)

    # Select one asset observation as a demonstration.
    asset_row = data.iloc[[0]].copy()

    # Model 1
    failure_probability = failure_model.predict_proba(
        asset_row
    )[:, 1][0]

    # Model 2
    predicted_rul = rul_model.predict(
        asset_row
    )[0]

    print("Failure probability:", round(failure_probability, 4))
    print("Predicted RUL:", round(predicted_rul, 2), "hours")

else:
    print(
        "Set DATA_PATH to your dataset and rerun this cell."
    )


## 8. Full Asset Lifecycle Recommendation

The following cell combines ML predictions with business/operational attributes.

Update the values from your actual asset record.


In [ ]:
# ============================================================
# 9. FULL LIFECYCLE RECOMMENDATION
# ============================================================

# Example values — replace with values from the asset record.
criticality = "High"
repair_cost = 35000
hours_since_maintenance = 420
previous_failures = 2

if "failure_probability" in globals() and "predicted_rul" in globals():

    recommendation = lifecycle_decision(
        failure_probability=failure_probability,
        rul_hours=predicted_rul,
        criticality=criticality,
        repair_cost=repair_cost,
        hours_since_maintenance=hours_since_maintenance,
        previous_failures=previous_failures
    )

    print("=" * 55)
    print("        INTELLIGENT ASSET LIFECYCLE REPORT")
    print("=" * 55)
    print(f"Failure Probability : {failure_probability:.2%}")
    print(f"Predicted RUL       : {predicted_rul:.2f} hours")
    print(f"Criticality         : {criticality}")
    print(f"Repair Cost         : ₹{repair_cost:,.0f}")
    print(f"Risk Score          : {recommendation['risk_score']}/100")
    print(f"Recommended Action  : {recommendation['action']}")
    print(f"Reason              : {recommendation['reason']}")
    print("=" * 55)

else:
    print("Run the prediction cell first.")


## 9. Risk Distribution

This gives the project a dashboard-friendly output: every asset can be assigned a numerical risk score and lifecycle action.


In [ ]:
# ============================================================
# 10. VISUALIZE SCENARIO RISK
# ============================================================

plt.figure(figsize=(8, 5))

plt.bar(
    scenario_results["Asset"],
    scenario_results["Risk Score"]
)

plt.xlabel("Asset")
plt.ylabel("Risk Score")
plt.title("Asset Lifecycle Risk Score")
plt.ylim(0, 100)

for i, value in enumerate(scenario_results["Risk Score"]):
    plt.text(
        i,
        value + 2,
        str(value),
        ha="center"
    )

plt.show()


# Final Architecture

```text
                    SENSOR / ASSET DATA
                            |
             +--------------+--------------+
             |                             |
             v                             v
       MODEL 1: RF                  MODEL 2: RF
     Failure Probability              RUL Hours
             |                             |
             +--------------+--------------+
                            |
                            v
                  LIFECYCLE ENGINE
                            |
        +-------------------+-------------------+
        |                   |                   |
        v                   v                   v
   Criticality        Maintenance          Repair Cost
                      History
        |                   |                   |
        +-------------------+-------------------+
                            |
                            v
                  RISK SCORE + REASON
                            |
                            v
        MONITOR / MAINTAIN / REPAIR /
             REPLACE / UPGRADE
```

This is the component that turns the two predictive models into an **Intelligent Asset Lifecycle Management** system.
